In [1]:
# 04_callaway_santanna.ipynb
# Heterogeneity-robust difference-in-differences (Callaway & Sant'Anna, 2021) using
# an acute health shock (first new hospitalization) as an absorbing treatment.
# Reports overall and dynamic ATT for CHE/OOP, a pre-trend check, and the
# private-insurance buffer via a baseline-PHI sample split, benchmarked against a
# naive two-way FE event study. Figure is grayscale, no titles, 600-dpi PNG and PDF.

import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS
from differences import ATTgt

PROC, FIG, TAB = "../data/processed", "../results/figures", "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "savefig.dpi": 600,
                     "figure.dpi": 110, "grid.color": "0.9"})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

p = pd.read_parquet(f"{PROC}/khp_panel.parquet")
p["che40f"] = p["che40"].astype(float); p["che25f"] = p["che25"].astype(float)

def run_cs(outcome, split=None):
    cols = ["PIDWON", "year", "cohort", outcome] + (["base_phi"] if split else [])
    d = p[cols].dropna(subset=[outcome]).copy()
    d["cohort"] = d["cohort"].replace(0, np.nan)      # never-treated must be missing for CS
    d = d.set_index(["PIDWON", "year"])
    att = ATTgt(data=d, cohort_column="cohort", base_period="universal")
    att.fit(formula=outcome, est_method="dr", control_group="not_yet_treated",
            sample_split_column=split, progress_bar=False)
    return att

# ---- overall and dynamic ATT for three outcomes ----
simple_rows, event_rows = [], []
cs_che40_event = None
for out, lab in [("che40f", "CHE40"), ("che25f", "CHE25"), ("ihs_oop", "asinh_OOP")]:
    att = run_cs(out)
    s = att.aggregate("simple"); s.columns = [str(c) for c in s.columns]
    simple_rows.append({"outcome": lab, "ATT": float(s.iloc[0, 0]), "se": float(s.iloc[0, 1])})
    e = att.aggregate("event"); e.columns = [str(c) for c in e.columns]
    for rp, r in e.iterrows():
        event_rows.append({"outcome": lab, "k": int(rp), "ATT": float(r.iloc[0]),
                           "se": (float(r.iloc[1]) if not pd.isna(r.iloc[1]) else np.nan)})
    if out == "che40f":
        cs_che40_event = {int(rp): (float(r.iloc[0]), float(r.iloc[1]) if not pd.isna(r.iloc[1]) else 0.0)
                          for rp, r in e.iterrows()}
pd.DataFrame(simple_rows).to_csv(f"{TAB}/t04_cs_overall_att.csv", index=False)
pd.DataFrame(event_rows).to_csv(f"{TAB}/t04_cs_event_att.csv", index=False)

# ---- PHI buffer: CS overall ATT split by baseline insurance ----
att_s = run_cs("che40f", split="base_phi")
spl = att_s.aggregate("simple"); spl.columns = [str(c) for c in spl.columns]
buf_rows = [{"group": str(nm), "ATT": float(r.iloc[0]), "se": float(r.iloc[1])} for nm, r in spl.iterrows()]
buf = pd.DataFrame(buf_rows); buf.to_csv(f"{TAB}/t04_cs_phi_buffer.csv", index=False)

# ---- naive TWFE event study on the same acute shock, for comparison ----
p["k"] = np.where(p["cohort"] > 0, p["year"] - p["cohort"], np.nan)
def kbin(k):
    return np.nan if pd.isna(k) else max(-4, min(4, int(k)))
p["kb"] = p["k"].apply(kbin)
evk = [-4, -3, -2, 0, 1, 2, 3, 4]
for kk in evk:
    p[f"e{kk}"] = ((p["cohort"] > 0) & (p["kb"] == kk)).astype(int)
cols = [f"e{kk}" for kk in evk]
cnt = p.groupby("PIDWON")["year"].transform("size")
es = p[cnt >= 2].set_index(["PIDWON", "year"])
d = es[["che40f"] + cols].dropna()
tw = PanelOLS(d["che40f"], d[cols], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
twc = {kk: (tw.params[f"e{kk}"], tw.std_errors[f"e{kk}"]) for kk in evk}; twc[-1] = (0.0, 0.0)

# ---- Figure: (A) TWFE vs CS dynamic CHE40, (B) PHI buffer overall ATT ----
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ks = [-4, -3, -2, -1, 0, 1, 2, 3, 4]
tv = np.array([twc[k][0] for k in ks]); ts = np.array([twc[k][1] for k in ks])
cv = np.array([cs_che40_event.get(k, (np.nan, 0))[0] for k in ks]); cs = np.array([cs_che40_event.get(k, (0, 0))[1] for k in ks])
ax[0].axhline(0, color="0.5", lw=0.8); ax[0].axvline(-0.5, color="0.7", ls="--", lw=1)
ax[0].errorbar(np.array(ks) - 0.08, tv, yerr=1.96 * ts, fmt="s--", color="0.6", ms=5, capsize=3, lw=1.3, label="Two-way FE")
ax[0].errorbar(np.array(ks) + 0.08, cv, yerr=1.96 * cs, fmt="o-", color="0.12", ms=6, capsize=3, lw=1.8,
               markerfacecolor="white", markeredgewidth=1.4, label="Callaway-Sant'Anna")
ax[0].set_xlabel("Years relative to first hospitalization (k)"); ax[0].set_ylabel("Delta P(CHE 40%) vs k=-1")
ax[0].set_xticks(ks); ax[0].legend(frameon=False)
labs = ["No PHI", "Has PHI"]; vals = [buf.iloc[0]["ATT"] * 100, buf.iloc[1]["ATT"] * 100]
ses = [buf.iloc[0]["se"] * 100, buf.iloc[1]["se"] * 100]
bars = ax[1].bar(labs, vals, yerr=[1.96 * s for s in ses], color=["0.25", "0.65"], edgecolor="0.3", capsize=5, width=0.6)
for b, v in zip(bars, vals):
    ax[1].text(b.get_x() + b.get_width() / 2, v + 0.25, f"+{v:.1f}pp", ha="center", fontweight="bold")
ax[1].set_ylabel("Overall ATT on P(CHE 40%), pp"); ax[1].set_xlabel("Baseline private insurance")
save_fig(fig, "fig06_robustness_cs")

print("CS overall ATT:")
print(pd.DataFrame(simple_rows).round(4).to_string(index=False))
print("CS PHI buffer (CHE40):")
print(buf.round(4).to_string(index=False))


CS overall ATT:
  outcome    ATT     se
    CHE40 0.0361 0.0029
    CHE25 0.0688 0.0042
asinh_OOP 0.9271 0.0773
CS PHI buffer (CHE40):
       group    ATT     se
base_phi = 0 0.0723 0.0074
base_phi = 1 0.0259 0.0031
